In [233]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import pickle
import tensorflow as tf
tf.config.run_functions_eagerly(True)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Dropout, Reshape, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
import tensorflow.keras.backend as K
from tensorflow.keras.layers import GRU, BatchNormalization

In [215]:
#loaidmng our train dataset
train = pd.read_csv("train_cl.csv")
with open("train_cols.pkl", "rb") as f:
    train_cols = pickle.load(f)

In [216]:
#checking to see if we have any ,issing values
train.isnull().any().sum()

0

In [217]:
#splitting our data into Independent and dependent variables
X = train.drop(columns=['Sales'])
y = train['Sales']
# y_log = y

In [ ]:

cat_cols = ['Store', 'StoreType', 'Assortment', 'PromoInterval', 'StateHoliday']
seq_cols = [col for col in X.columns if 'Sales_Lag' in col or 'Sales_RollingMean' in col]
num_cols = [col for col in X.columns if col not in cat_cols + seq_cols]


In [ ]:
#splititng the data on train and validation sets on 45 days
days_in_val = 45  
num_stores = train['Store'].nunique()
val_rows = days_in_val * num_stores

val_start= len(X) - val_rows


X_train = X.iloc[:val_start].copy()
y_train= y.iloc[:val_start].copy()
X_val = X.iloc[val_start:].copy()
y_val = y.iloc[val_start:].copy()


print(f"Total rows in X: {len(X)}")
print(f"Row count calculated for Val: {val_rows}")
print(f"Train size: {len(X_train)}, Validation size: {len(X_val)}")

Total rows in X: 844338
Row count calculated for Val: 50175
Train size: 794163, Validation size: 50175


In [ ]:
#scaling the dataset
# Loading  scaler that was saved during preprocessing
with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

cols_to_scale = [
    'CompetitionDistance', 'CompetitionOpenMonths', 'Promo2SinceWeek',
    'Promo2SinceYear', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Lag_28',
    'Sales_RollingMean_7', 'Sales_RollingMean_28', 'Year', 'Day',
    'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos',
    'DayOfYear_sin', 'DayOfYear_cos', 'Month'
]

# Transform train and validation using the SAME scaler
X_train[cols_to_scale] = scaler.transform(X_train[cols_to_scale])
X_val[cols_to_scale] = scaler.transform(X_val[cols_to_scale])



In [ ]:
from sklearn.preprocessing import LabelEncoder
cat_cols = ['Store', 'StoreType', 'Assortment', 'PromoInterval', 'StateHoliday']
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_val[col] = X_val[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    label_encoders[col] = le

In [ ]:
#combining numeric and categorical data
X_train_scaled = pd.concat([X_train[num_cols + seq_cols], X_train[cat_cols]], axis=1).values
X_val_scaled = pd.concat([X_val[num_cols + seq_cols], X_val[cat_cols]], axis=1).values

In [224]:
 #defining our metric of success RMSPE
def rmspe(y_true, y_pred):
    pct_error = (y_true - y_pred) / K.clip(y_true, K.epsilon(), None)
    return K.sqrt(K.mean(K.square(pct_error)))


Modeling

DNN

In [ ]:
#implementing early stopping for patience 10 and reducing learning rate on plateau,they were both implemented because the previous models would run for close to 2 hrs
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.00001, verbose=1)
adam_optimizer = Adam(0.001)

In [ ]:
#buikding out model and compiling the model
num_features = X_train_scaled.shape[1]

dnn = Sequential([
    Dense(256, activation='relu', input_dim= num_features),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='linear')
])
dnn.compile(
    optimizer=adam_optimizer,
    loss='mse',
    metrics=[rmspe]
)



/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#fitting our model 
#gettinng a callback to save the best weights
mc_dnn = ModelCheckpoint("dnn.weights.h5", save_best_only=True, save_weights_only=True)

dnn.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=512,
    callbacks=[es, mc_dnn, reduce_lr],
    verbose=1
)

/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


Epoch 1/50
   1/1552 ━━━━━━━━━━━━━━━━━━━━ 4:19 167ms/step - loss: 66.1890 - rmspe: 0.9274

/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1552/1552 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - loss: 2.5808 - rmspe: 0.1226

/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1552/1552 ━━━━━━━━━━━━━━━━━━━━ 65s 42ms/step - loss: 0.7605 - rmspe: 0.0788 - val_loss: 107.1901 - val_rmspe: 1.1866 - learning_rate: 0.0010
Epoch 2/50
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 62s 40ms/step - loss: 0.2288 - rmspe: 0.0550 - val_loss: 662.6899 - val_rmspe: 2.9458 - learning_rate: 0.0010
Epoch 3/50
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 62s 40ms/step - loss: 0.1590 - rmspe: 0.0461 - val_loss: 162.8306 - val_rmspe: 1.4620 - learning_rate: 0.0010
Epoch 4/50
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 0.1399 - rmspe: 0.0433
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 62s 40ms/step - loss: 0.1366 - rmspe: 0.0428 - val_loss: 238.2600 - val_rmspe: 1.7686 - learning_rate: 0.0010
Epoch 5/50
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 63s 40ms/step - loss: 0.1230 - rmspe: 0.0406 - val_loss: 38.9583 - val_rmspe: 0.7162 - learning_rate: 5.0000e-04
Epoch 6/50
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 62s 40ms/step - loss: 0.1162 - rmspe: 0.0395 - val_loss

In [ ]:
#model evaluation 
dnn.load_weights("dnn.weights.h5")
print("DNN RMSPE:", dnn.evaluate(X_val_scaled, y_val, verbose=0)[1])

/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


DNN RMSPE: 0.03083135560154915


LSTM

In [ ]:
#preping data for lstm and gru by defining the time steps
time_steps = 1
train_seq = X_train_scaled.reshape(-1, time_steps, num_features)
val_seq= X_val_scaled.reshape(-1, time_steps, num_features)


In [ ]:
#model architecture and compilation
lstm = Sequential([
    LSTM(128, activation='relu', input_shape=(time_steps, num_features)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

lstm.compile(
    optimizer=adam_optimizer,
    loss='mse',
    metrics=[rmspe]
)

/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
#fitting our model
mc_lstm = ModelCheckpoint("lstm.weights.h5", save_best_only=True, save_weights_only=True)
lstm.fit(
    train_seq, y_train,
    validation_data=(val_seq, y_val),
    epochs=50,
    batch_size=512,
    callbacks=[es, mc_lstm, reduce_lr],
    verbose=1
)


In [ ]:
#model performance
lstm.load_weights("lstm.weights.h5")
print("LSTM RMSPE:", lstm.evaluate(val_seq, y_val, verbose=0)[1])

/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


LSTM RMSPE: 0.18506112694740295


GRU

In [ ]:
#gru model architecture 
gru = Sequential([
    GRU(128, activation='relu', input_shape=(TIME_STEPS, INPUT_DIM)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dense(32, activation='relu'),
    Dense(1)
])

gru.compile(
    optimizer=adam_optimizer,
    loss='mse',
    metrics=[rmspe]
)

/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
#model training
mc_gru = ModelCheckpoint("gru.weights.h5", save_best_only=True, save_weights_only=True)
gru.fit(
    train_seq, y_train,
    validation_data=(val_seq, y_val),
    epochs=50,
    batch_size=512,
    callbacks=[es, mc_gru, reduce_lr],
    verbose=1

)

Epoch 1/50
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 5.7248 - rmspe: 0.8708 - val_loss: 0.2561 - val_rmspe: 0.3866 - learning_rate: 0.0010
Epoch 2/50
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.2614 - rmspe: 0.6853 - val_loss: 0.3584 - val_rmspe: 0.4136 - learning_rate: 0.0010
Epoch 3/50
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.2365 - rmspe: 0.6203 - val_loss: 0.8723 - val_rmspe: 1.6617 - learning_rate: 0.0010
Epoch 4/50
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.3749 - rmspe: 0.9202
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.4120 - rmspe: 0.9809 - val_loss: 2.5287 - val_rmspe: 1.0803 - learning_rate: 0.0010
Epoch 5/50
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.2002 - rmspe: 0.5585 - val_loss: 0.8772 - val_rmspe: 0.5841 - learning_rate: 5.0000e-04
Epoch 6/50
1552/1552 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.2572 - rmspe: 0.6483 - val_lo

In [ ]:
#gru model evaluation
gru.load_weights("gru.weights.h5")
print("GRU RMSPE:", gru.evaluate(val_seq, y_val, verbose=0)[1])


GRU RMSPE: 0.22028109431266785
